In [ ]:
import torch
from matplotlib import pyplot as plt

torch.set_printoptions(edgeitems=2, linewidth=75)

# Operating hours
hours = torch.tensor([
    -3.0,
    -2.0,
    -1.0,
     0.0,
     1.0,
     2.0,
     3.0
])


# Real electricity consumption
consumption = torch.tensor([
    14.0,
     7.0,
     4.0,
     5.0,
    10.0,
    19.0,
    32.0
])

def model(x, w2, w1, b):
    return w2 * x**2 + w1 * x + b

def loss_fn(predicted, real):
    return ((predicted - real)**2).mean()

def dloss_fn(predicted, real):
    return 2 * (predicted - real) / predicted.size(0)

def dmodel_dw2(x):
    return x**2

def dmodel_dw1(x):
    return x

def dmodel_db():
    return 1.0

def grad_fn(x, real, predicted):
    dloss_dpredicted = dloss_fn(predicted, real)

    dloss_dw2 = dloss_dpredicted * dmodel_dw2(x)
    dloss_dw1 = dloss_dpredicted * dmodel_dw1(x)
    dloss_db = dloss_dpredicted * dmodel_db()

    return torch.stach([
        dloss_dw2.sum(),
        dloss_dw1.sum(),
        dloss_db()
    ])

def training_loop(
    n_epochs,
    learning_rate,
    params,
    x,
    real
):
    for epoch in range(1, n_epochs + 1):
        w2, w1, b = params

        predicted = model(x, w2, w1, b)

        loss = loss_fn(predicted, real)
        grad = grad_fn(x, real, predicted)
        params = params - learning_rate * grad

        if epoch in {
            1,
            2,
            3,
            10,
            100,
            1000,
            5000
        }:
            print(f"Epoch {epoch}, " f"Loss {float(loss):.6f}, " f"Params: {params}")
    return params


params = training_loop(
    n_epochs=5000,
    learning_rate=1e-2,
    params=torch.tensor([0.0, 0.0, 0.0]),
    x=hours,
    real=consumption
)

predicted_consumption = model(hours, *params)

print("\nLearned parameters:")
print(f"w2 = {params[0]:.4f}")
print(f"w1 = {params[1]:.4f}")
print(f"b  = {params[2]:.4f}")

plt.figure(dpi=150)
plt.xlabel("Operating Hours")
plt.ylabel("Electricity Consumption")

plt.plot(hours.numpy(), predicted_consumption.detach().numpy(), label="Polynomial Prediction")
plt.plot(hours.numpy(), consumption.numpy(), "o", label="Real Data")

plt.legend()
plt.show()